# Health Transformation Lab coding-agent tasks: quickstart

Load the [coding-agent task set](https://www.kaggle.com/datasets/jcblearning/health-transformation-lab-coding-agent-tasks)
and look at one task the way an agent would: its problem statement, the fix and the tests, the repository at the
task's base commit, and the code graph and embeddings to search it.

Each task is a real change from the history of
[github.com/healthtransformationlab/health-transformation-lab-nlp](https://github.com/healthtransformationlab/health-transformation-lab-nlp),
in the layout of the Gemma 4 Developer Agent competition's dataset. Only tasks whose tests fail without the fix and
pass with it are included. A personal learning project, not affiliated with Kaggle, Google, the competition's
organisers or any other organization. Runs on CPU, no internet.

In [ ]:
import glob
import json
import os
import tarfile
from collections import Counter
from pathlib import Path

import numpy as np

# the attached dataset on Kaggle; AGENT_TASKS_DIR points elsewhere (a local copy)
found = glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)
ROOT = Path(os.environ.get("AGENT_TASKS_DIR") or (Path(found[0]).parent if found else "."))
tasks = [json.loads(line) for line in (ROOT / "tasks.jsonl").read_text().splitlines() if line.strip()]
print(f"{len(tasks)} tasks in {ROOT}")

## One task: the problem, the fix and the tests

In [ ]:
task = tasks[0]
print(task["instance_id"], "at", task["repo"], task["base_commit"][:12])
print("\n--- problem statement ---\n" + task["problem_statement"][:1500])
print("\n--- the fix (patch) ---\n" + task["patch"][:2000])
print("\n--- the tests (test_patch) ---\n" + task["test_patch"][:2000])

## The repository at the task's base commit

In [ ]:
workspace = Path("workspace") / task["instance_id"]
workspace.mkdir(parents=True, exist_ok=True)
with tarfile.open(ROOT / "snapshots" / f"{task['instance_id']}.tgz") as tf:
    try:
        tf.extractall(workspace, filter="data")   # Python 3.12+: refuse links and paths outside the folder
    except TypeError:
        tf.extractall(workspace)
print(sorted(p.name for p in workspace.iterdir()))

## Its code graph and embeddings: what is near a symbol?

In [ ]:
name = f"{task['repo'].split('/')[-1]}_{task['base_commit']}"
graph = json.loads((ROOT / "graphs" / f"{name}.json").read_text())   # NetworkX node-link JSON
nodes, edges = graph["nodes"], graph["edges"]
print(len(nodes), "functions, classes and methods;", len(edges), "calls")

vectors = np.load(ROOT / "embeddings" / f"{name}.npz")
ids = list(vectors.files)
matrix = np.stack([vectors[i] for i in ids])
matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
calls = Counter(e["source"] for e in edges)
symbol = max((n["id"] for n in nodes), key=lambda i: calls[i])   # calls the most others; pick any id
scores = matrix @ matrix[ids.index(symbol)]
nearest = [(ids[i], float(scores[i])) for i in np.argsort(-scores) if ids[i] != symbol][:5]
print(f"nearest to {symbol}:")
for node, score in nearest:
    print(f"  {score:.3f}  {node}")